In [1]:
# Data Cleaning and Standardisation

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, DoubleType

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 4, Finished, Available, Finished, False)

In [15]:
inter_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/CRM/interactions/interactions'
cust_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/CRM/customers/customers'
inventory_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/ERP/inventory/inventory_20260518.csv'
product_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/ERP/products/products_20260518.csv'
shipment_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/LOGISTICS/shipments/shipments.csv'
trans_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/POS/transactions/transactions'
review_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/SOCIAL-MEDIA/reviews/REVIEW'
store_path = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Bronze/POS/stores/stores.csv'

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 17, Finished, Available, Finished, False)

In [4]:
silver = 'abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver'

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 6, Finished, Available, Finished, False)

In [5]:
def write_silver(df, table_name: str):
    path = f"{silver}/{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(path)
    )
    print(f"[silver] wrote {table_name} → {path}  ({df.count()} rows)")
    df.unpersist()

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 7, Finished, Available, Finished, False)

In [6]:
#Customer entity
df_customer = spark.read.format('delta').load(cust_path)
customers_silver = (
    df_customer
    # cast numerics
    .withColumn("age", F.col("age").cast(IntegerType()))
    # parse date
    .withColumn("signup_date", F.to_date("signup_date", "yyyy-MM-dd"))
    # normalise strings
    .withColumn("full_name",     F.trim(F.col("full_name")))
    .withColumn("gender",        F.trim(F.upper(F.col("gender"))))
    .withColumn("city",          F.trim(F.col("city")))
    .withColumn("country",       F.trim(F.col("country")))
    .withColumn("loyalty_tier",  F.trim(F.col("loyalty_tier")))
    # fill missing loyalty tier
    .fillna({"loyalty_tier": "Bronze"})
    # drop rows missing critical keys
    .dropna(subset=["customer_id"])
    # deduplicate (keep first occurrence)
    .dropDuplicates(["customer_id"])
    # add audit column
    .withColumn("_silver_load_ts", F.current_timestamp())
)
write_silver(customers_silver, "Customers")


StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 8, Finished, Available, Finished, False)

[silver] wrote Customers → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver/Customers  (10000 rows)


In [16]:
#Customer Interaction Entity
df_interaction = spark.read.format('delta').load(inter_path)

interactions_silver = (
    df_interaction
    .withColumn("interaction_id",   F.col("interaction_id").cast(IntegerType()))
    .withColumn("interaction_date", F.to_date("interaction_date", "yyyy-MM-dd"))
    .withColumn("interaction_type", F.trim(F.col("interaction_type")))
    .withColumn("agent_name",       F.trim(F.col("agent_name")))
    .dropna(subset=["interaction_id", "customer_id"])
    .dropDuplicates(["interaction_id"])
    .withColumn("_silver_load_ts", F.current_timestamp())
)

write_silver(interactions_silver, "Interactions")

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 18, Finished, Available, Finished, False)

[silver] wrote Interactions → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver/Interactions  (10000 rows)


In [8]:
#Stores Entity
df_stores = spark.read.format("csv").option("header",True).load(store_path)
stores_silver = (
    df_stores
    .withColumn("store_name", F.trim(F.col("store_name")))
    .withColumn("city",       F.trim(F.col("city")))
    .withColumn("region",     F.trim(F.col("region")))
    .dropna(subset=["store_id"])
    .dropDuplicates(["store_id"])
    .withColumn("_silver_load_ts", F.current_timestamp())
)

write_silver(stores_silver, "Stores")

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 10, Finished, Available, Finished, False)

[silver] wrote Stores → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver/Stores  (30 rows)


In [9]:
#Product Entity
df_product = spark.read.format("csv").option("header",True).load(product_path)
products_silver = (
    df_product
    .withColumn("unit_price",      F.col("unit_price").cast(DoubleType()))
    .withColumn("stock_quantity",  F.col("stock_quantity").cast(IntegerType()))
    .withColumn("product_name",    F.trim(F.col("product_name")))
    .withColumn("category",        F.trim(F.col("category")))
    .withColumn("brand",           F.trim(F.col("brand")))
    # guard: negative prices / quantities are data errors
    .filter(F.col("unit_price") > 0)
    .filter(F.col("stock_quantity") >= 0)
    .dropna(subset=["product_id"])
    .dropDuplicates(["product_id"])
    .withColumn("_silver_load_ts", F.current_timestamp())
)

write_silver(products_silver, "Products")


StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 11, Finished, Available, Finished, False)

[silver] wrote Products → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver/Products  (2000 rows)


In [10]:
#Inventory Entity 
df_inventory = spark.read.format("csv").option("header",True).load(inventory_path)
inventory_silver = (
    df_inventory
    .withColumn("stock_received",   F.col("stock_received").cast(IntegerType()))
    .withColumn("stock_dispatched", F.col("stock_dispatched").cast(IntegerType()))
    .withColumn("inventory_date",   F.to_timestamp("inventory_date", "yyyy-MM-dd HH:mm:ss.SSSSSSS"))
    # derive net stock movement for the period
    .withColumn("net_stock", F.col("stock_received") - F.col("stock_dispatched"))
    .dropna(subset=["warehouse_id", "product_id", "inventory_date"])
    .withColumn("_silver_load_ts", F.current_timestamp())
)

write_silver(inventory_silver, "Inventory")

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 12, Finished, Available, Finished, False)

[silver] wrote Inventory → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver/Inventory  (50000 rows)


In [11]:
#Transactions Entity
df_trans = spark.read.format("delta").load(trans_path)
trans_silver = (
    df_trans
    .withColumn("quantity",     F.col("quantity").cast(IntegerType()))
    .withColumn("sales_amount", F.col("sales_amount").cast(DoubleType()))
    .withColumn("transaction_timestamp",
                F.to_timestamp("transaction_timestamp", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("transaction_date",
                F.to_date("transaction_timestamp"))
    .withColumn("sales_channel",  F.trim(F.col("sales_channel")))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    # drop rows without a transaction or customer id
    .dropna(subset=["transaction_id", "customer_id", "product_id"])
    # remove zero / negative amounts
    .filter(F.col("sales_amount") > 0)
    .filter(F.col("quantity") > 0)
    .dropDuplicates(["transaction_id"])
    .withColumn("_silver_load_ts", F.current_timestamp())
)

write_silver(trans_silver, "Transactions")

StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 13, Finished, Available, Finished, False)

[silver] wrote Transactions → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver/Transactions  (85 rows)


In [12]:
#Shipment Entity
df_shipment = spark.read.format("csv").option("header","true").load(shipment_path)
shipments_silver = (
    df_shipment
    .withColumn("ship_date",     F.to_date("ship_date",     "yyyy-MM-dd"))
    .withColumn("delivery_date", F.to_date("delivery_date", "yyyy-MM-dd"))
    .withColumn("weight_kg",     F.col("weight_kg").cast(DoubleType()))
    .withColumn("shipping_cost", F.col("shipping_cost").cast(DoubleType()))
    # recalculate delivery_days from dates where it is null
    .withColumn("delivery_days",
        F.when(
            F.col("delivery_days").isNotNull(),
            F.col("delivery_days").cast(IntegerType())
        ).otherwise(
            F.datediff(F.col("delivery_date"), F.col("ship_date"))
        )
    )
    # normalise status
    .withColumn("status", F.trim(F.col("status")))
    .dropna(subset=["shipment_id", "customer_id", "product_id"])
    .dropDuplicates(["shipment_id"])
    .withColumn("_silver_load_ts", F.current_timestamp())
)

write_silver(shipments_silver, "Shipments")



StatementMeta(, dd01d0a0-6cda-4cd5-b323-452eb79b33f2, 14, Finished, Available, Finished, False)

[silver] wrote Shipments → abfss://RBusiness360_Fabric_Project@onelake.dfs.fabric.microsoft.com/RBusiness360_LakeHouse.Lakehouse/Files/Silver/Shipments  (25000 rows)
